# 🛍️ Lumière EU E-Commerce — End-to-End Analytics Pipeline
### AIDVS Executive Master · Porto Business School · Group Project

**Business question:** *"Where should Lumière focus its commercial attention over the next two quarters to grow profitably?"*

| Detail | Value |
|---|---|
| Dataset | 82 000 orders · 5 200 customers · 177 products · 6 150 returns |
| Period | Jan 2024 – Dec 2025 |
| Markets | 8 EU countries · 3 channels |
| Currency | EUR (€) |

---

## Notebook Structure

| Phase | Section | Description |
|---|---|---|
| 1 | Setup | Imports, config, constants |
| 2 | Data Enrichment | Date dims, order metrics, RFM, CLV, country KPIs |
| 3 | KPI Framework | Commercial + Marketing KPIs with formulas |
| 4 | Visualisations | Matplotlib charts for all key KPIs |
| 5 | Executive Storytelling | Strategic narrative and recommendations |


---
## Phase 0 — Setup
*Imports, datasets.*

In [1]:
import pandas as pd
import numpy as np


# ── Upload the datasets ────────────────────────────────────────────────────────────────────
orders = pd.read_parquet("../data/cleaned/orders_clean.parquet")
products = pd.read_parquet("../data/cleaned/products_clean.parquet")
customers = pd.read_parquet("../data/cleaned/customers_clean.parquet")
returns = pd.read_parquet("../data/cleaned/returns_clean.parquet")
targets = pd.read_parquet("../data/cleaned/targets_clean.parquet")


print("✅  Setup complete")


✅  Setup complete


In [2]:
# Verify the df
orders.info()
products.info()
customers.info()
returns.info()
targets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82000 entries, 0 to 81999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order ID        82000 non-null  object        
 1   Order Date      82000 non-null  datetime64[ns]
 2   Ship Date       82000 non-null  datetime64[ns]
 3   Customer ID     82000 non-null  object        
 4   Product ID      82000 non-null  object        
 5   Country         82000 non-null  object        
 6   Channel         82000 non-null  object        
 7   Quantity        82000 non-null  int64         
 8   Unit Price      82000 non-null  float64       
 9   Discount        82000 non-null  float64       
 10  Shipping Cost   82000 non-null  float64       
 11  Payment Method  82000 non-null  object        
 12  Shipping Days   82000 non-null  int64         
dtypes: datetime64[ns](2), float64(3), int64(2), object(6)
memory usage: 8.1+ MB
<class 'pandas.core.frame.Data

---
## Phase 1 — Data Enrichment
*Build the analytical layer: order-level metrics, date dimensions, product KPIs, RFM segmentation, country growth, and retention rates.*


### 1.1  FactTable

In [3]:
# Merge product cost data into orders to create a FactTable: 'orders_enriched'
orders_enriched = orders.merge(
    products[
        [
            "Product ID",
            "Unit Cost",
            "List Price",
            "Category",
            "Sub-Category",
            "Brand",
            "Product Margin Pct"
        ]
    ],
    on="Product ID",
    how="left"
)

In [4]:
orders_enriched.head()

,Order ID,Order Date,Ship Date,Customer ID,Product ID,Country,Channel,Quantity,Unit Price,Discount,Shipping Cost,Payment Method,Shipping Days,Unit Cost,List Price,Category,Sub-Category,Brand,Product Margin Pct
0,EU-2024-000001,2024-01-01,2024-01-05,C14312,P1128,France,Web,1,148.86,0.2,4.9,Credit Card,4,58.12,148.86,Home & Living,Bedding,Nordic Light,0.609566
1,EU-2024-000002,2024-01-01,2024-01-04,C10318,P1025,France,Mobile App,2,164.90,0.2,0.0,Credit Card,3,84.69,164.90,Apparel,Outerwear,Maison Lumière,0.486416
2,EU-2024-000003,2024-01-01,2024-01-03,C14095,P1045,Germany,Mobile App,1,235.53,0.2,0.0,Credit Card,2,117.90,235.53,Apparel,Dresses,Atelier 21,0.499427
3,EU-2024-000004,2024-01-01,2024-01-08,C11990,P1020,Italy,Mobile App,2,142.07,0.2,0.0,PayPal,7,70.16,142.07,Apparel,Bottoms,Lumière Studio,0.506159
4,EU-2024-000005,2024-01-01,2024-01-06,C11643,P1054,Germany,Mobile App,1,423.66,0.2,0.0,Bank Transfer,5,136.48,423.66,Footwear,Sneakers,Atelier 21,0.677855


In [5]:
orders_enriched.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82000 entries, 0 to 81999
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Order ID            82000 non-null  object        
 1   Order Date          82000 non-null  datetime64[ns]
 2   Ship Date           82000 non-null  datetime64[ns]
 3   Customer ID         82000 non-null  object        
 4   Product ID          82000 non-null  object        
 5   Country             82000 non-null  object        
 6   Channel             82000 non-null  object        
 7   Quantity            82000 non-null  int64         
 8   Unit Price          82000 non-null  float64       
 9   Discount            82000 non-null  float64       
 10  Shipping Cost       82000 non-null  float64       
 11  Payment Method      82000 non-null  object        
 12  Shipping Days       82000 non-null  int64         
 13  Unit Cost           82000 non-null  float64   

### 1.2 Revenue Metrics

In [6]:
# Revenue metrics & margin

# Gross Revenue = Unit Price * Quantity
orders_enriched["Gross Revenue"]   = orders_enriched["Unit Price"]   * orders_enriched["Quantity"]

# Discount Amount = Gross Revenue * Discount
orders_enriched["Discount Amount"] = orders_enriched["Gross Revenue"] * orders_enriched["Discount"]

# Net Revenue = Gross Revenue - Discount Amount
orders_enriched["Net Revenue"]     = orders_enriched["Gross Revenue"] - orders_enriched["Discount Amount"]

# COGS = Unit Cost * Quantity
orders_enriched["COGS"]            = orders_enriched["Unit Cost"]     * orders_enriched["Quantity"]

# Gross Profit = Net Revenue - COGS
orders_enriched["Gross Profit"]    = orders_enriched["Net Revenue"]   - orders_enriched["COGS"]

# Margin Pct = Avg Order Value 
orders_enriched["Margin Pct"]      = np.where(orders_enriched["Net Revenue"] > 0,
                                       orders_enriched["Gross Profit"] / orders_enriched["Net Revenue"], 0)
# Join the margins with the Fact Table
print(f"Total Gross Revenue : €{orders_enriched['Gross Revenue'].sum():>14,.0f}")
print(f"Total Net Revenue   : €{orders_enriched['Net Revenue'].sum():>14,.0f}")
print(f"Total Gross Profit  : €{orders_enriched['Gross Profit'].sum():>14,.0f}")
print(f"Blended Margin %    :  {orders_enriched['Gross Profit'].sum()/orders_enriched['Net Revenue'].sum()*100:.1f}%")
print(f"Avg Order Value     : €{orders_enriched['Net Revenue'].mean():>14,.2f}")

Total Gross Revenue : €    22,235,447
Total Net Revenue   : €    20,217,136
Total Gross Profit  : €    10,700,421
Blended Margin %    :  52.9%
Avg Order Value     : €        246.55


### 1.3. Date dimensions

In [26]:
# Create a Date Dimension from the Order Date column in the orders_enriched df

DimDate = pd.to_datetime(orders_enriched["Order Date"], errors="coerce") # Convert Order Date to datetime format, coercing errors to NaT

# Create a Date Dimension from the Order Date column in the orders_enriched df
DimDate = pd.DataFrame({
    "Order Date": pd.to_datetime(orders_enriched["Order Date"], errors="coerce")
})

DimDate["Year"]          = DimDate["Order Date"].dt.year
DimDate["Quarter"]       = DimDate["Order Date"].dt.quarter
DimDate["Month"]         = DimDate["Order Date"].dt.month
DimDate["Month Name"]    = DimDate["Order Date"].dt.strftime("%B")
DimDate["Week"]          = DimDate["Order Date"].dt.isocalendar().week.astype(int)
DimDate["Year Month"]    = DimDate["Order Date"].dt.to_period("M").astype(str)
DimDate["Year Quarter"]  = DimDate["Order Date"].dt.year.astype(str) + "-Q" + DimDate["Order Date"].dt.quarter.astype(str)
DimDate["Fiscal Year"]   = np.where(DimDate["Order Date"].dt.month >= 7, DimDate["Order Date"].dt.year + 1, DimDate["Order Date"].dt.year)
DimDate["Fiscal Quarter"]= ((DimDate["Order Date"].dt.month - 7) % 12 // 3 + 1)

print("Date dimensions added:")
print(DimDate[["Order Date","Year","Quarter","Month","Year Month",
                "Year Quarter","Fiscal Year","Fiscal Quarter"]].head(3).to_string())

Date dimensions added:
  Order Date  Year  Quarter  Month Year Month Year Quarter  Fiscal Year  Fiscal Quarter
0 2024-01-01  2024        1      1    2024-01      2024-Q1         2024               3
1 2024-01-01  2024        1      1    2024-01      2024-Q1         2024               3
2 2024-01-01  2024        1      1    2024-01      2024-Q1         2024               3


In [8]:
# Save the date dimension as a separate parquet file for use in the next notebook
DimDate.to_parquet("../data/curated/dim_date.parquet", index=False)

In [9]:
# Save the enriched orders dataset for future use
orders_enriched.to_parquet("../data/curated/orders_enriched.parquet", index=False) 
orders_enriched

,Order ID,Order Date,Ship Date,Customer ID,Product ID,Country,Channel,Quantity,Unit Price,Discount,...,Category,Sub-Category,Brand,Product Margin Pct,Gross Revenue,Discount Amount,Net Revenue,COGS,Gross Profit,Margin Pct
0,EU-2024-000001,2024-01-01,2024-01-05,C14312,P1128,France,Web,1,148.86,0.20,...,Home & Living,Bedding,Nordic Light,0.609566,148.86,29.7720,119.0880,58.12,60.9680,0.511958
1,EU-2024-000002,2024-01-01,2024-01-04,C10318,P1025,France,Mobile App,2,164.90,0.20,...,Apparel,Outerwear,Maison Lumière,0.486416,329.80,65.9600,263.8400,169.38,94.4600,0.358020
2,EU-2024-000003,2024-01-01,2024-01-03,C14095,P1045,Germany,Mobile App,1,235.53,0.20,...,Apparel,Dresses,Atelier 21,0.499427,235.53,47.1060,188.4240,117.90,70.5240,0.374284
3,EU-2024-000004,2024-01-01,2024-01-08,C11990,P1020,Italy,Mobile App,2,142.07,0.20,...,Apparel,Bottoms,Lumière Studio,0.506159,284.14,56.8280,227.3120,140.32,86.9920,0.382699
4,EU-2024-000005,2024-01-01,2024-01-06,C11643,P1054,Germany,Mobile App,1,423.66,0.20,...,Footwear,Sneakers,Atelier 21,0.677855,423.66,84.7320,338.9280,136.48,202.4480,0.597319
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81995,EU-2025-081996,2025-12-31,2026-01-05,C14199,P1037,France,Web,1,202.19,0.15,...,Apparel,Outerwear,Maison Lumière,0.494288,202.19,30.3285,171.8615,102.25,69.6115,0.405044
81996,EU-2025-081997,2025-12-31,2026-01-02,C10228,P1055,France,Web,3,136.33,0.10,...,Footwear,Sneakers,Sole & Co.,0.619013,408.99,40.8990,368.0910,155.82,212.2710,0.576681
81997,EU-2025-081998,2025-12-31,2026-01-04,C10886,P1076,Italy,Mobile App,4,68.85,0.00,...,Footwear,Formal,Sole & Co.,0.542774,275.40,0.0000,275.4000,125.92,149.4800,0.542774
81998,EU-2025-081999,2025-12-31,2026-01-03,C11175,P1028,Ireland,Web,5,42.42,0.00,...,Apparel,Outerwear,Lumière Studio,0.478076,212.10,0.0000,212.1000,110.70,101.4000,0.478076


### 1.4.  RFM segmentation
* 1. Customer Aggregation
* 2. RFM Scoring
* 3. Create RFM Scoring
* 4. Segmement Customers

* RFM segmentation is a behavioral data analysis method used by marketers to group customers based on their past purchase history. 
* It evaluates three key metrics to determine which customers are the most valuable, engaged, or at risk of churning:
    * Recency: How recently a customer made a purchase.
    * Frequency: How often a customer makes a purchase.
    * Monetary Value: How much money the customer spends

customer_analytics = (
    orders_enriched
    .groupby("Customer ID")
    .agg(
        Revenue=("Net Revenue","sum"),
        Orders=("Order ID","nunique"),
        Profit=("Gross Profit","sum")
    )
)

In [10]:
# Snapshot date in base of the most recent order date in the dataset
SNAPSHOT_DATE = pd.Timestamp("2026-01-01") # latest order date in the dataset is 2025-12-31, so we set the snapshot date to the next day to calculate recency correctly

# RFM Analysis
rfm = (
    orders_enriched
    .groupby("Customer ID")
    .agg(
        # Recency is calculated as the number of days since the most recent order date for each customer
        Recency=("Order Date",
                 lambda x: (SNAPSHOT_DATE - x.max()).days), 
        # lambda is used to calculate recency as the difference between the snapshot date and the most recent order date

        # Frequency is calculated as the number of unique orders per customer
        Frequency=("Order ID", "nunique"),

        # Monetary is calculated as the total net revenue per customer
        Monetary=("Net Revenue", "sum")
    )
    .reset_index()
)

rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,C10001,23,52,14510.040
1,C10002,26,6,1529.347
2,C10003,3,30,8333.294
3,C10004,10,24,6358.664
4,C10005,30,11,3186.147


In [11]:
# RFM Score
rfm["R"]= pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)
rfm["F"]= pd.qcut(
    rfm["Frequency"],
    5,
    labels=[1, 2, 3, 4, 5]
)
rfm["M"]= pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1, 2, 3, 4, 5]
)

In [12]:
# Create RFM Scoring 
rfm["RFM Score"] = (
    rfm["R"].astype(int) +
    rfm["F"].astype(int) +
    rfm["M"].astype(int)
)

# check distribution of RFM scores
rfm["RFM Score"].value_counts().sort_index()
# value_counts is used to count the number of occurrences of each RFM score, and sort_index is used to sort the scores in ascending order for better readability

RFM Score
3     411
4     382
5     391
6     467
7     443
8     422
9     400
10    385
11    373
12    360
13    418
14    345
15    385
Name: count, dtype: int64

Why number 1 and 2 in the RFM Score are not counted? 

* Because the RFM score is the sum of three components (R, F, M) each ranging from 1 to 5, 
* the most common scores will be around the middle of the distribution (e.g., 7, 8, 9) rather than the extremes (1 or 15). 
* Scores of 1 or 2 would indicate very low recency, frequency, and monetary value, which are less common among customers.

In [13]:
# 4. Segmentation Customers
def segment(score):
    if score >= 13:
        return "Champions"
    elif score >= 10:
        return "Loyal Customers"
    elif score >= 7:
        return "Potential Loyalists"
    elif score >= 4:
        return "Needs Attention"
    else:
        return "At Risk"
    
# Apply the segmentation function to the RFM Score column
rfm["Segment"] = rfm["RFM Score"].apply(segment) 
# Apply is used to apply the segment function to each value in the RFM Score column and create a new Segment column with the corresponding segment labels

# View 
rfm[["RFM Score", "Segment"]]

,RFM Score,Segment
0,13,Champions
1,6,Needs Attention
2,15,Champions
3,12,Loyal Customers
4,9,Potential Loyalists
...,...,...
5177,9,Potential Loyalists
5178,6,Needs Attention
5179,15,Champions
5180,3,At Risk


In [14]:
rfm["Segment"].value_counts()

Segment
Potential Loyalists    1265
Needs Attention        1240
Champions              1148
Loyal Customers        1118
At Risk                 411
Name: count, dtype: int64

### 1.5.  CLV - Customer Lifetime Value
* It is a metric that estimates the total net profit or revenue a business can expect from a single customer throughout their entire relationship with the company.
* Understanding CLV helps you shift focus from one-time transactions to long-term profitability. It allows you to:
    * Optimize Acquisition: Helps calculate how much you can afford to spend on marketing to acquire a new customer (known as CAC). 
    * A standard business benchmark is a CLV to CAC ratio of 3:1.
    * Improve Retention: Identifies your most valuable customer segments so you can prioritize loyalty programs, upselling, and customer service resources.
    * Forecast Revenue: Provides a lens into long-term business health and expected future cash flows.

In [15]:
clv = (
    orders_enriched.groupby("Customer ID")
    .agg(
        Revenue=("Net Revenue", "sum"),
        Orders=("Order ID", "nunique"),
    )
)

# clv 
clv["CLV_Estimate"] = clv["Revenue"] * 1.5 # simple CLV estimate based on total revenue multiplied by a retention factor of 1.5

clv.head()

,Revenue,Orders,CLV_Estimate
Customer ID,,,
C10001,14510.040,52,21765.0600
C10002,1529.347,6,2294.0205
C10003,8333.294,30,12499.9410
C10004,6358.664,24,9537.9960
C10005,3186.147,11,4779.2205


In [16]:
# Join CLV with RFM segments
customer_analytics = rfm.merge(clv, on="Customer ID", how="left")

# Saved the enriched dataset in the curated folder for future use
customer_analytics.to_parquet("../data/curated/customer_analytics.parquet", index=False)

customer_analytics

,Customer ID,Recency,Frequency,Monetary,R,F,M,RFM Score,Segment,Revenue,Orders,CLV_Estimate
0,C10001,23,52,14510.0400,3,5,5,13,Champions,14510.0400,52,21765.06000
1,C10002,26,6,1529.3470,3,1,2,6,Needs Attention,1529.3470,6,2294.02050
2,C10003,3,30,8333.2940,5,5,5,15,Champions,8333.2940,30,12499.94100
3,C10004,10,24,6358.6640,4,4,4,12,Loyal Customers,6358.6640,24,9537.99600
4,C10005,30,11,3186.1470,3,3,3,9,Potential Loyalists,3186.1470,11,4779.22050
...,...,...,...,...,...,...,...,...,...,...,...,...
5177,C15196,188,14,4080.5880,1,4,4,9,Potential Loyalists,4080.5880,14,6120.88200
5178,C15197,31,8,1682.9680,2,2,2,6,Needs Attention,1682.9680,8,2524.45200
5179,C15198,4,47,13726.1005,5,5,5,15,Champions,13726.1005,47,20589.15075
5180,C15199,180,3,450.6660,1,1,1,3,At Risk,450.6660,3,675.99900


## 2. KPIs

### 2.1.  Retention metrics

In [35]:
# Add Year to the orders_enriched dataset for future use in time-based analyses
orders_enriched["Year"] = orders_enriched["Order Date"].dt.year
orders_enriched["Year-Month"] = orders_enriched["Order Date"].dt.to_period("M").astype(str)
orders_enriched.head()

,Order ID,Order Date,Ship Date,Customer ID,Product ID,Country,Channel,Quantity,Unit Price,Discount,...,Gross Revenue,Discount Amount,Net Revenue,COGS,Gross Profit,Margin Pct,Year,Disc_Tier,Year Month,Year-Month
0,EU-2024-000001,2024-01-01,2024-01-05,C14312,P1128,France,Web,1,148.86,0.2,...,148.86,29.772,119.088,58.12,60.968,0.511958,2024,11–20%,2024-01,2024-01
1,EU-2024-000002,2024-01-01,2024-01-04,C10318,P1025,France,Mobile App,2,164.90,0.2,...,329.80,65.960,263.840,169.38,94.460,0.358020,2024,11–20%,2024-01,2024-01
2,EU-2024-000003,2024-01-01,2024-01-03,C14095,P1045,Germany,Mobile App,1,235.53,0.2,...,235.53,47.106,188.424,117.90,70.524,0.374284,2024,11–20%,2024-01,2024-01
3,EU-2024-000004,2024-01-01,2024-01-08,C11990,P1020,Italy,Mobile App,2,142.07,0.2,...,284.14,56.828,227.312,140.32,86.992,0.382699,2024,11–20%,2024-01,2024-01
4,EU-2024-000005,2024-01-01,2024-01-06,C11643,P1054,Germany,Mobile App,1,423.66,0.2,...,423.66,84.732,338.928,136.48,202.448,0.597319,2024,11–20%,2024-01,2024-01


In [24]:
# Customer Retention Analysis
active_2024 = set(orders_enriched[orders_enriched["Year"]==2024]["Customer ID"])
active_2025 = set(orders_enriched[orders_enriched["Year"]==2025]["Customer ID"])
retained    = active_2024 & active_2025

freq = orders_enriched.groupby("Customer ID")["Order ID"].nunique()
repeat_rate = (freq >= 2).mean()

print(f"Unique customers 2024      : {len(active_2024):,}")
print(f"Unique customers 2025      : {len(active_2025):,}")
print(f"Customer growth %          : +{(len(active_2025)-len(active_2024))/len(active_2024)*100:.1f}%")
print(f"Retained (2024 → 2025)     : {len(retained):,}")
print(f"Retention rate             : {len(retained)/len(active_2024):.1%}")
print(f"Churn rate                 : {1-len(retained)/len(active_2024):.1%}")
print(f"Repeat purchase rate       : {repeat_rate:.1%}")

# Save Customer Retention Analysis results for future use 
customer_retention_analysis = pd.DataFrame({
    "Metric": ["Unique Customers 2024", "Unique Customers 2025", "Customer Growth %", 
               "Retained (2024 → 2025)", "Retention Rate", "Churn Rate", "Repeat Purchase Rate"],
    
    "Value": [f"{len(active_2024):,}", f"{len(active_2025):,}", f"+{(len(active_2025)-len(active_2024))/len(active_2024)*100:.1f}%",
              f"{len(retained):,}", f"{len(retained)/len(active_2024):.1%}", f"{1-len(retained)/len(active_2024):.1%}", f"{repeat_rate:.1%}"]
})
customer_retention_analysis.to_parquet("../data/curated/customer_retention_analysis.parquet", index=False)
customer_retention_analysis

Unique customers 2024      : 3,247
Unique customers 2025      : 5,127
Customer growth %          : +57.9%
Retained (2024 → 2025)     : 3,192
Retention rate             : 98.3%
Churn rate                 : 1.7%
Repeat purchase rate       : 98.6%


,Metric,Value
0,Unique Customers 2024,"3,247"
1,Unique Customers 2025,"5,127"
2,Customer Growth %,+57.9%
3,Retained (2024 → 2025),"3,192"
4,Retention Rate,98.3%
5,Churn Rate,1.7%
6,Repeat Purchase Rate,98.6%


### 2.2.  Commercial / Merchandising KPIs

In [ ]:
# Revenue Metrics & Margins
total_gross_rev  = orders_enriched["Gross Revenue"].sum()
total_disc_amt   = orders_enriched["Discount Amount"].sum()
total_net_rev    = orders_enriched["Net Revenue"].sum()
total_cogs       = orders_enriched["COGS"].sum()
total_profit     = orders_enriched["Gross Profit"].sum()
margin_pct       = total_profit / total_net_rev
avg_discount_pct = total_disc_amt / total_gross_rev
aov              = orders_enriched["Net Revenue"].mean()
returned_orders = returns["Order ID"].nunique()
return_rate = returned_orders / len(orders_enriched)

kpis_commercial = pd.DataFrame({
    "KPI":    ["Net Revenue","Gross Profit","Gross Margin %","Avg Order Value",
               "Avg Discount %","Return Rate","Total Orders","Unique Products"],
    "Value":  [f"€{total_net_rev:,.0f}", f"€{total_profit:,.0f}",
               f"{margin_pct:.1%}", f"€{aov:,.2f}",
               f"{avg_discount_pct:.1%}", f"{return_rate:.1%}",
               f"{len(orders_enriched):,}", 
               f"{orders_enriched['Product ID'].nunique()}"],
    "Formula":["Unit Price × Qty × (1−Discount)",
               "Net Revenue − COGS",
               "(Net Rev − COGS) / Net Rev",
               "Net Revenue / Order Count",
               "Discount Amount / Gross Revenue",
               "Returned Orders / Total Orders",
               "COUNT(DISTINCT order_id)",
               "COUNT(DISTINCT product_id)"],
})
display(kpis_commercial)

# save the kpis commercial to parquet for future use   
kpis_commercial.to_parquet("../data/curated/kpis_commercial.parquet", index=False)



,KPI,Value,Formula
0,Net Revenue,"€20,217,136",Unit Price × Qty × (1−Discount)
1,Gross Profit,"€10,700,421",Net Revenue − COGS
2,Gross Margin %,52.9%,(Net Rev − COGS) / Net Rev
3,Avg Order Value,€246.55,Net Revenue / Order Count
4,Avg Discount %,9.1%,Discount Amount / Gross Revenue
5,Return Rate,7.5%,Returned Orders / Total Orders
6,Total Orders,"82,000",COUNT(DISTINCT order_id)
7,Unique Products,177,COUNT(DISTINCT product_id)


Revenue breakdown by category and brand

### 2.3  Discount impact on margin

In [ ]:
# Discount Impact Analysis
# Create discount tiers based on the Discount column in the orders_enriched dataset

orders_enriched["Disc_Tier"] = pd.cut(
    orders_enriched["Discount"],
    bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5], # bins are used to define the discount tiers based on the discount percentage, with the specified breakpoints
    labels=["No discount",
            "1–10%",
            "11–20%",
            "21–30%",
            "31–50%"] 
            # labels are used to assign descriptive names to the discount tiers based on the defined bins
)
disc_impact = orders_enriched.groupby("Disc_Tier", observed=True).agg( 
    Orders      = ("Order ID","count"),
    Net_Revenue = ("Net Revenue","sum"),
    Margin_Pct  = ("Margin Pct","mean"),
).round(3)
display(disc_impact)


,Orders,Net_Revenue,Margin_Pct
Disc_Tier,,,
No discount,29048,7903749.100,0.566
1–10%,30675,7676297.168,0.529
11–20%,15557,3433617.838,0.469
21–30%,4856,915386.892,0.382
31–50%,1864,288084.710,0.253


### 2.4  Marketing / Customer KPIs

In [49]:
kpis_marketing = pd.DataFrame({
    "KPI": ["Unique Customers",
            "New Customers 2025",
            "Retention Rate",
            "Churn Rate",
            "Repeat Purchase Rate",
            "Champions Share",
            "At-Risk Share",
            "Avg CLV (all)",
            "Avg CLV (Champions)"],
    
    "Value": [
        f"{customer_analytics['Customer ID'].nunique():,}",                                                 # total unique customers across all years
        f"{len(active_2025 - active_2024):,}",                                                              # new customers in 2025 who were not active in 2024
        f"{len(retained)/len(active_2024):.1%}",                                                            # retention rate calculated as the proportion of customers active in 2024 who were retained in 2025
        f"{1-len(retained)/len(active_2024):.1%}",                                                          # churn rate calculated as the complement of the retention rate
        f"{repeat_rate:.1%}",                                                                               # repeat purchase rate calculated as the proportion of customers with 2 or more orders
        f"{(customer_analytics['Segment']=='Champions').mean():.1%}",                                       # champions share calculated as the proportion of customers in the Champions segment
        f"{(customer_analytics['Segment']=='At Risk').mean():.1%}",                                         # at-risk share calculated as the proportion of customers in the At-Risk segment
        f"€{customer_analytics['CLV_Estimate'].mean():,.0f}",                                               # average CLV estimate across all customers
        f"€{customer_analytics[customer_analytics['Segment']=='Champions']['CLV_Estimate'].mean():,.0f}",   # average CLV estimate for customers in the Champions segment
    ],
})

# save the kpis marketing to parquet for future use
kpis_marketing.to_parquet("../data/curated/kpis_marketing.parquet", index=False)

display(kpis_marketing)

,KPI,Value
0,Unique Customers,"5,182"
1,New Customers 2025,"1,935"
2,Retention Rate,98.3%
3,Churn Rate,1.7%
4,Repeat Purchase Rate,98.6%
5,Champions Share,22.2%
6,At-Risk Share,7.9%
7,Avg CLV (all),"€5,852"
8,Avg CLV (Champions),"€13,389"


### 2.5  Country KPIs — YoY growth & target achievement

In [29]:
targets.head()

,Country,Year-Month,Target Revenue,Year,Month
0,Germany,2024-01,61272.727273,2024,1
1,Germany,2024-02,63636.363636,2024,2
2,Germany,2024-03,86153.846154,2024,3
3,Germany,2024-04,91851.851852,2024,4
4,Germany,2024-05,104807.692308,2024,5


In [ ]:
# YoY revenue per country
yr = orders_enriched.groupby(["Country","Year"])["Net Revenue"].sum().unstack(fill_value=0)
yr.columns = ["Rev_2024","Rev_2025"]

# Calculate YoY growth percentage for each country
yr["YoY_Growth_Pct"] = (yr["Rev_2025"] - yr["Rev_2024"]) / yr["Rev_2024"] * 100

# Target achievement (full period)
monthly_actual = (
    orders_enriched.groupby(["Country","Year-Month"])["Net Revenue"]
    .sum().reset_index()
)
monthly_actual.columns = ["Country","Year-Month","Actual"]

merged = monthly_actual.merge(targets[["Country","Year-Month","Target Revenue"]],
                               on=["Country","Year-Month"])

# Target achievement by country is calculated as the total actual revenue divided by the total target revenue for each country, expressed as a percentage
agg = merged.groupby("Country").agg(
    Actual = ("Actual", "sum"),
    Target = ("Target Revenue", "sum")
)
achievement = (
    (agg["Actual"] / agg["Target"] * 100)
    .rename("Target_Achievement_Pct")
    .reset_index()
)

country_kpis = yr.reset_index().merge(achievement, on="Country")
country_kpis["Rev_per_Customer"] = country_kpis["Rev_2025"] / (
    orders_enriched[orders_enriched["Year"]==2025].groupby("Country")["Customer ID"].nunique().values
)

display(country_kpis.sort_values("Rev_2025", ascending=False).round(1))

,Country,Rev_2024,Rev_2025,YoY_Growth_Pct,Target_Achievement_Pct,Rev_per_Customer
2,Germany,1334557.6,3457847.0,159.1,92.3,3004.2
1,France,1162769.8,2807779.4,141.5,87.4,2799.4
7,Spain,959472.4,2217973.1,131.2,91.7,2691.7
4,Italy,875063.3,2126799.1,143.0,92.5,2794.7
5,Netherlands,397347.3,1080441.8,171.9,68.2,2701.1
6,Portugal,377114.9,956584.7,153.7,82.1,2813.5
0,Belgium,301675.2,916027.1,203.6,86.6,2844.8
3,Ireland,339107.3,906575.7,167.3,95.9,2780.9


In [ ]:
# Save the country KPIs to parquet for future use
country_kpis.to_parquet("../data/curated/country_kpis.parquet", index=False)

---
## Phase 3 — Executive Storytelling
### *"Where should Lumière focus its commercial attention over the next two quarters to grow profitably?"*


In [48]:
# ── Headline numbers for the narrative ───────────────────────────────────────
yoy_growth = (orders_enriched[orders_enriched["Year"]==2025]["Net Revenue"].sum() /
              orders_enriched[orders_enriched["Year"]==2024]["Net Revenue"].sum() - 1)
champ_count         = (customer_analytics["Segment"]=="Champions").sum()
at_risk_count       = (customer_analytics["Segment"]=="At Risk").sum()
deep_disc_orders    = (orders_enriched["Discount"] >= 0.3).sum()
deep_disc_margin    = orders_enriched[orders_enriched["Discount"]>=0.3]["Margin Pct"].mean()
no_disc_margin      = orders_enriched[orders_enriched["Discount"]==0]["Margin Pct"].mean()

print(f"2025 YoY Revenue Growth     : +{yoy_growth:.0%}")
print(f"Champions                   : {champ_count:,} customers")
print(f"At-Risk                     : {at_risk_count:,} customers")
print(f"Orders with 30%+ discount   : {deep_disc_orders:,}")
print(f"Margin at 30%+ discount     : {deep_disc_margin:.1%}")
print(f"Margin at no discount       : {no_disc_margin:.1%}")
print(f"Margin compression          : {(no_disc_margin-deep_disc_margin):.1%} pp")


2025 YoY Revenue Growth     : +152%
Champions                   : 1,148 customers
At-Risk                     : 411 customers
Orders with 30%+ discount   : 6,720
Margin at 30%+ discount     : 34.6%
Margin at no discount       : 56.6%
Margin compression          : 22.0% pp


### Key Findings

1. **Revenue is growing fast but unevenly.** 2025 net revenue is 2.5× the 2024 level. Germany (€4.8M) and France (€4.0M) account for 44% of total. Netherlands reached only 68.2% of its target — the worst in the portfolio.

2. **Margins are healthy but discount-sensitive.** Blended gross margin is 52.9%. Orders with 30%+ discounts see margin fall to ~26–39%, a 31pp compression vs undiscounted orders.

3. **North Coast and Lumière Maison are underutilised.** North Coast (55.3% margin) and Lumière Maison (59% margin — highest in portfolio) are structurally more profitable but underrepresented in revenue mix.

4. **Customer base is sticky.** 98.3% retention 2024→2025. However, 411 customers (7.9%) are At-Risk. The Champions cohort (1,148 customers) generates disproportionate CLV and is primed for a loyalty programme.

---

### Strategic Recommendations

| # | Recommendation | Priority | Quarter | Expected Impact |
|---|---|---|---|---|
| 1 | **Germany + Accessories focus** — highest revenue × margin combination | 🔴 High | Q3 2026 | +€800K net revenue |
| 2 | **Eliminate 30%+ discounts** except planned clearance — replace with loyalty perks | 🔴 High | Q3 2026 | +€180K gross profit |
| 3 | **Elevate North Coast & Lumière Maison** in Open-to-Buy allocation | 🟡 Medium | Q4 2026 | +1pp blended margin |
| 4 | **Win-back 411 At-Risk customers** with 3-touch re-engagement sequence | 🟡 Medium | Q4 2026 | €250K–380K CLV recovered |
| 5 | **Investigate Netherlands** before increasing investment (68.2% achievement) | 🟡 Medium | Q3 2026 | Risk mitigation |
| 6 | **Launch Champions loyalty programme** — free express shipping, early access | 🟡 Medium | Q4 2026 | +15–20% frequency |

---

### Executive Summary

> Lumière has proven product-market fit — 52.9% gross margins, 98% customer retention, and 2.5× revenue growth demonstrate a healthy DTC model. Profitable scaling over the next two quarters requires three disciplines:
>
> **Q3 2026:** Concentrate commercial investment on Germany and Accessories. Initiate a discount policy review that eliminates 30%+ promotional depth except for planned end-of-season clearance.
>
> **Q4 2026:** Activate the customer base — Champions loyalty programme to increase frequency among the top 29.9% of customers; structured win-back campaign for 411  At-Risk customers. Shift brand mix toward North Coast and Lumière Maison for structural margin improvement.
>
> **Geography:** Maintain Germany and France investment. Pause incremental Netherlands spend pending market diagnostic. Belgium (+204% YoY growth) is the highest-upside emerging market and deserves a dedicated H2 2026 budget.
